In [ ]:
import pandas as pd
import numpy as np

inference_df = pd.read_csv("../data/raw/inference.csv")
inference_df.head()

In [ ]:
inference_df.dtypes

In [18]:
inference_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71205 entries, 0 to 71204
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   index                71205 non-null  int64 
 1   store_ID             71205 non-null  int64 
 2   day_of_week          71205 non-null  int64 
 3   date                 71205 non-null  object
 4   nb_customers_on_day  71205 non-null  int64 
 5   open                 71205 non-null  int64 
 6   promotion            71205 non-null  int64 
 7   school_holiday       71205 non-null  int64 
 8   state_holiday_0      71205 non-null  bool  
 9   state_holiday_a      71205 non-null  bool  
 10  state_holiday_b      71205 non-null  bool  
 11  state_holiday_c      71205 non-null  bool  
dtypes: bool(4), int64(7), object(1)
memory usage: 4.6+ MB


In [ ]:
inference_df.describe()

In [ ]:
inference_df.isna().sum()

In [ ]:
inference_df.nunique()

In [ ]:
inference_df["store_ID"]

# we will remove unamed, date
# we change the format of state_holiday


In [ ]:
inference_df["state_holiday"].value_counts()


In [ ]:
# Which columns contain the features of the websites?
#X = sale_df.drop(columns=['URL', 'Type'])
#y = websites['Type']
#display(X)

inference_df = pd.get_dummies(inference_df, columns=['state_holiday'])

In [ ]:
inference_df.info()

In [ ]:
X = inference_df.drop(columns=["index","state_holiday_0","date","open"])

display(X)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
#plt.figure(figsize=(12, 10))
#sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm")
#plt.show()


features = X.select_dtypes(include="number")
mask = np.triu(np.ones_like(features.corr(), dtype=bool)) 
corr = features.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True,cmap="Blues", vmin=-1, vmax=1,mask=mask)
plt.title("Numeric Feature Correlations")
plt.show()

In [ ]:
X = X.drop(columns=["open"])
display(X)

In [ ]:
features = X.select_dtypes(include="number")
mask = np.triu(np.ones_like(features.corr(), dtype=bool)) 
corr = features.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True,cmap="Blues", vmin=-1, vmax=1,mask=mask)
plt.title("Numeric Feature Correlations")
plt.show()

### we dropped open feature because it has strong correlation with the day_of_week and nb_customers_on_day
data doesn't have any empty samples in it

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)
print("Train:", X_train.shape) 
print("Validation:", X_val.shape) 
print("Test:", X_test.shape)

In [ ]:
X.info()

In [ ]:
X.describe()

In [ ]:
#from sklearn.linear_model import LinearRegression 
#model = LinearRegression() 
#model.fit(X_train, y_train) 
#predictions = model.predict(X_test)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(np.min(X_train_scaled))
print(np.max(X_train_scaled))


In [ ]:


print("Means:")
print(X_train_scaled.mean(axis=0))

print("Standard deviations:")
print(X_train_scaled.std(axis=0))

scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

print(scaler.mean_)
print(scaler.scale_)



In [ ]:
scaled_df.head()

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_scaled, y_train)

predictions = model.predict(X_test_scaled)



In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, predictions)

print(f"MAE:  {mae:.2f}")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")

In [ ]:
results = pd.DataFrame({
    "Actual": y_test,
    "Predicted": predictions
})

display(results.head(10))

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(y_test, predictions, alpha=0.3)
plt.xlabel("Actual sales")
plt.ylabel("Predicted sales")
plt.title("Actual vs Predicted Sales")

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red"
)

plt.show()

In [ ]:
val_predictions = model.predict(X_val_scaled)

val_mae = mean_absolute_error(y_val, val_predictions)
val_rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
val_r2 = r2_score(y_val, val_predictions)

print(f"Validation MAE:  {val_mae:.2f}")
print(f"Validation RMSE: {val_rmse:.2f}")
print(f"Validation R²:   {val_r2:.4f}")

In [ ]:
test_predictions = model.predict(X_test_scaled)

print(f"Test MAE:  {mean_absolute_error(y_test, test_predictions):.2f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, test_predictions)):.2f}")
print(f"Test R²:   {r2_score(y_test, test_predictions):.4f}")